### Pip install que precisam ocorrer antes de importe de blibliotecas especificas

In [ ]:
!python -m ensurepip --upgrade
%pip install uv


!uv pip install torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu121
!uv pip install xformers --index-url https://download.pytorch.org/whl/cu121

!uv pip install torchao==0.7.0
!uv pip install unsloth unsloth-zoo

!uv pip install accelerate transformers trl peft bitsandbytes datasets
!uv pip install setuptools pandas scikit-learn google-generativeai matplotlib ipywidgets ollama

In [ ]:
# BLOCO 2
import os
import time
import torch
import pandas as pd
from datasets import Dataset
from sklearn.model_selection import train_test_split
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
import google.generativeai as genai

# Verificação de Hardware
device = "cuda" if torch.cuda.is_available() else "cpu"
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Nenhuma GPU'
print(f"Iniciando Pipeline em: {device} | Dispositivo: {gpu_name}")

In [ ]:
import subprocess
import time

# rode isso no terminal
# !curl -fsSL https://ollama.com/install.sh | sh
#!ollama pull qwen2.5:7b

process = subprocess.Popen(["ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(5) # Aguarda o servidor subir


In [ ]:
import os
import re
import pandas as pd
import ollama

MODELO_OLLAMA = "qwen2.5:7b"
FREQUENCIA_FAKES = 3 
ARQUIVO_ENTRADA = "./com_propostas.csv"
ARQUIVO_SAIDA = "./propostas_com_fakes.csv"

def chunk_texto(texto: str, max_chars: int = 1000) -> list[str]:
    if not isinstance(texto, str) or not texto.strip():
        return []

    paragrafos = [p.strip() for p in re.split(r"\n+", texto) if p.strip()]
    chunks = []
    chunk_atual = ""

    for p in paragrafos:
        if len(chunk_atual) + len(p) <= max_chars:
            chunk_atual += (" " if chunk_atual else "") + p
        else:
            if chunk_atual:
                chunks.append(chunk_atual)
            if len(p) > max_chars:
                frases = re.split(r"(?<=[.!?]) +", p)
                sub_chunk = ""
                for f in frases:
                    if len(sub_chunk) + len(f) <= max_chars:
                        sub_chunk += (" " if sub_chunk else "") + f
                    else:
                        if sub_chunk:
                            chunks.append(sub_chunk)
                        sub_chunk = f
                if sub_chunk:
                    chunks.append(sub_chunk)
                chunk_atual = ""
            else:
                chunk_atual = p

    if chunk_atual:
        chunks.append(chunk_atual)

    return chunks

def gerar_proposta_falsa(candidato: str, proposta_real: str) -> str:
    prompt = f"""Você é um gerador de dados sintéticos para treinamento de checagem de fatos.
Candidato: {candidato}
Proposta Real: {proposta_real}

Gere UMA proposta FALSA/DISTORCIDA que pareça ter sido dita pelo candidato "{candidato}", baseando-se na proposta real acima.
Aplique um exagero inviável, alteração de público-alvo ou inclusão de custos/regras absurdas.
Retorne APENAS o texto da proposta falsa."""

    try:
        response = ollama.chat(model=MODELO_OLLAMA, messages=[{'role': 'user', 'content': prompt}])
        return response['message']['content'].strip()
    except Exception as e:
        print(f"      [ERRO OLLAMA] Falha para {candidato}: {e}")
        return ""

df_original = pd.read_csv(ARQUIVO_ENTRADA, sep=";", encoding="latin1")
df_original.columns = df_original.columns.str.strip().str.upper()
df_original['INDEX_ORIGINAL'] = df_original.index 

if os.path.exists(ARQUIVO_SAIDA):
    df_existente = pd.read_csv(ARQUIVO_SAIDA, sep=";", encoding="utf-8-sig")
    if 'INDEX_ORIGINAL' in df_existente.columns:
        indices_processados = df_existente['INDEX_ORIGINAL'].unique()
        df_pendente = df_original[~df_original['INDEX_ORIGINAL'].isin(indices_processados)]
    else:
        df_pendente = df_original.copy()
else:
    df_pendente = df_original.copy()

In [ ]:
FAKES_POR_CANDIDATO = 10  # Defina aqui o número exato de fakes a gerar para CADA candidato

reais_registros = []
falsas_registros = []
linhas_processadas = 0

print(f"Total de linhas pendentes para processar: {len(df_pendente)}")

for _, row in df_pendente.iterrows():
    cand_nome = row.get("NM_CANDIDATO", row.get("NM_URNA_CANDIDATO", "Desconhecido"))
    prop_real_bruta = row.get("PROPOSTA", "")

    if pd.isna(prop_real_bruta) or not str(prop_real_bruta).strip():
        continue

    chunks = chunk_texto(str(prop_real_bruta), max_chars=1000)
    fakes_geradas_candidato = 0

    for idx, chunk in enumerate(chunks):
        reg_real = row.to_dict()
        reg_real["PROPOSTA"] = chunk
        reg_real["CHUNK_ID"] = idx + 1
        reg_real["LABEL_CATEGORY"] = "true"
        reais_registros.append(reg_real)

        if fakes_geradas_candidato < FAKES_POR_CANDIDATO:
            prop_falsa = gerar_proposta_falsa(str(cand_nome), chunk)

            if prop_falsa:
                fakes_geradas_candidato += 1
                print(f"[NOVA FAKE GERADA] Candidato(a): {cand_nome} (Fake {fakes_geradas_candidato}/{FAKES_POR_CANDIDATO})")
                
                reg_falso = row.to_dict()
                reg_falso["PROPOSTA"] = prop_falsa
                reg_falso["CHUNK_ID"] = idx + 1
                reg_falso["LABEL_CATEGORY"] = "false"
                falsas_registros.append(reg_falso)
            else:
                print(f"[FALHA] Nao gerou fake para: {cand_nome} (Tentativa no trecho {idx + 1})")

    linhas_processadas += 1

    if linhas_processadas % 10 == 0:
        print(f"Salvando progresso... {linhas_processadas} linhas processadas ate agora.")
        if reais_registros or falsas_registros:
            df_batch = pd.concat([pd.DataFrame(reais_registros), pd.DataFrame(falsas_registros)], ignore_index=True)
            write_header = not os.path.exists(ARQUIVO_SAIDA)
            df_batch.to_csv(ARQUIVO_SAIDA, sep=";", index=False, encoding="utf-8-sig", mode='a', header=write_header)
            reais_registros = []
            falsas_registros = []

if reais_registros or falsas_registros:
    df_batch = pd.concat([pd.DataFrame(reais_registros), pd.DataFrame(falsas_registros)], ignore_index=True)
    write_header = not os.path.exists(ARQUIVO_SAIDA)
    df_batch.to_csv(ARQUIVO_SAIDA, sep=";", index=False, encoding="utf-8-sig", mode='a', header=write_header)

print("Finalizado")

In [ ]:

max_seq_length = 4096
lora_rank = 32         

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
    dtype=None,
)
model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = lora_rank,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

In [ ]:
import re
import pandas as pd


def restaurar_texto_corrompido(texto):
    if not isinstance(texto, str):
        return ""

    # 1. Tenta reverter o erro de encoding latin1 -> utf-8
    try:
        texto = texto.encode("latin1").decode("utf-8")
    except (UnicodeEncodeError, UnicodeDecodeError):
        pass

    # 2. Remove marcadores visuais de PDF / tópicos (ex: ●, •, ■)
    texto = re.sub(r"[●•■â\x97\x8f]", " ", texto)

    # 3. Limpa espaços duplos e quebras repetidas
    texto = re.sub(r"\s+", " ", texto).strip()

    return texto


# Carrega o CSV gerado
df = pd.read_csv("./propostas_com_fakes.csv", sep=";", encoding="utf-8-sig")

# Aplica a restauração
df["PROPOSTA"] = df["PROPOSTA"].apply(restaurar_texto_corrompido)

# Remove linhas onde a proposta ficou curta demais ou cortada
df = df[df["PROPOSTA"].str.len() > 30].reset_index(drop=True)

# Salva o arquivo limpo
df.to_csv(
    "./propostas_com_fakes_LIMPO.csv", sep=";", index=False, encoding="utf-8-sig"
)
print("Dataset limpo e corrigido com sucesso!")

In [ ]:
# NOVO BLOCO 7 - Carregamento e Preparação do Dataset
import pandas as pd

def preparar_dataset_treino(caminho_arquivo="./propostas_com_fakes.csv"):
    print(f"Carregando dados de: {caminho_arquivo}...")
    
    # Carrega o arquivo gerado no bloco 5 (salvo com utf-8-sig)
    df_carregado = pd.read_csv(caminho_arquivo, sep=";", encoding="utf-8-sig")
    
    # Adiciona a constante da fonte do TSE
    FONTE_TSE = "https://dadosabertos.tse.jus.br/dataset/candidatos-2026"
    df_carregado["FONTE"] = FONTE_TSE
    
    # Remove eventuais linhas vazias que possam ter sido geradas
    df_carregado = df_carregado.dropna(subset=["PROPOSTA", "LABEL_CATEGORY"])
    
    # Embaralha os dados para o treinamento do modelo (ML)
    df_treino = df_carregado.sample(frac=1, random_state=42).reset_index(drop=True)
    
    print(f"Dataset pronto! Total de amostras para treino: {len(df_treino)}")
    print(f"Distribuição das classes:\n{df_treino['LABEL_CATEGORY'].value_counts()}")
    
    return df_treino

# Executa a função e cria a variável que o Bloco 8 utilizará
df_final_treino = preparar_dataset_treino("./propostas_com_fakes.csv")

In [ ]:
prompt_template = """### Instrução:
Classifique o texto a seguir atribuído ao candidato {candidato} (Fonte: {fonte}) como 'verdadeiro' (true) ou 'falso' (false).

### Candidato:
{candidato}

### Fonte:
{fonte}

### Conteúdo / Proposta:
{proposta}

### Resposta:
{resposta}"""


def formatar_prompts(dataframe):
    textos = []
    for _, row in dataframe.iterrows():
        candidato = row.get("NM_CANDIDATO")
        if pd.isna(candidato) or not str(candidato).strip():
            candidato = row.get("NM_URNA_CANDIDATO", "Desconhecido")
        fonte = row.get("FONTE", "Não informada")
        proposta = row.get("PROPOSTA", "")

        texto = (
            prompt_template.format(
                candidato=candidato,
                fonte=fonte,
                proposta=proposta,
                resposta=row["LABEL_CATEGORY"],
            )
            + tokenizer.eos_token
        )
        textos.append(texto)
    return pd.DataFrame({"text": textos})


# Divisão de Treino e Validação (80/20)
train_df, val_df = train_test_split(
    df_final_treino, test_size=0.2, random_state=42, stratify=df_final_treino["LABEL_CATEGORY"]
)

dataset_treino = Dataset.from_pandas(formatar_prompts(train_df))
dataset_validacao = Dataset.from_pandas(formatar_prompts(val_df))

In [ ]:
import sys
import torch
import transformers.utils.import_utils

# 1. Desativa a checagem de versão do PyTorch dentro da biblioteca transformers
transformers.utils.import_utils.check_torch_load_is_safe = lambda *args, **kwargs: None

# Aplica a desativação em todos os módulos do transformers já importados na memória
for mod_name, mod in list(sys.modules.items()):
    if mod_name.startswith("transformers") and hasattr(mod, "check_torch_load_is_safe"):
        setattr(mod, "check_torch_load_is_safe", lambda *args, **kwargs: None)

# Imports do TRL e Trainer
from trl import SFTConfig, SFTTrainer

# Reduz o dataset de validação para acelerar as pausas de avaliação (1.000 amostras)
dataset_validacao_reduzido = dataset_validacao.select(
    range(min(1000, len(dataset_validacao)))
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset_treino,
    eval_dataset = dataset_validacao_reduzido,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = SFTConfig(
        per_device_train_batch_size = 4,   
        gradient_accumulation_steps = 4,     
        warmup_steps = 10,
        num_train_epochs = 2,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        
        # --- Configuração Otimizada de Validação ---
        eval_strategy = "steps",
        eval_steps = 250,
        save_strategy = "steps",
        save_steps = 250,
        save_total_limit = 2,
        load_best_model_at_end = True,
        metric_for_best_model = "eval_loss",
        greater_is_better = False,
        # -------------------------------------------
        
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "outputs_propostas",
    ),
)

# Retoma o treinamento diretamente do checkpoint-X
#caminho_checkpoint = "outputs_propostas/checkpoint-X" caso tenha parada apos um checkpoint
#print(f"Retomando treinamento a partir de: {caminho_checkpoint}...")
#trainer_stats = trainer.train(resume_from_checkpoint=caminho_checkpoint)

#sem checkpoint
trainer_stats = trainer.train()
print("Treinamento finalizado com sucesso!")

In [ ]:
gguf_directory = "modelo_propostas_gguf_v4"
quantization_method = "q4_k_m"
print(f"Exportando modelo para GGUF em quantização {quantization_method}...")

model.save_pretrained_gguf(
    gguf_directory, 
    tokenizer, 
    quantization_method = quantization_method
)

print(f"Modelo GGUF salvo com sucesso na pasta: ./{gguf_directory}")